# Notebook 10: Literature-based fixes for overfitting

Notebooks 08–09 showed that ResNet18S overfits mainly as **over-confidence** (validation loss and ECE rise while macro-F1 keeps improving),
and that a smaller network (ResNet10S) overfits less but is significantly less accurate. This notebook tries the fixes from the literature
that are cheap enough to run now. Everything uses the official split (`splits.csv`) and the same `base` recipe as R62–R64, so the results
compare directly with Notebook 08.

| # | Fix | Reference | Training? | Runs |
|---|---|---|---|---|
| 1 | **Temperature scaling** of ResNet18S (single and 3-seed ensemble): one parameter T fitted on the validation set | Guo et al., ICML 2017 | no | – |
| 2 | **Mixup**, α = 0.2: train on blends of two images and their labels | Zhang et al., ICLR 2018; calibration: Thulasidasan et al., NeurIPS 2019 | yes | R131–R133 |
| 3 | **CutMix**, α = 1.0, p = 0.5: paste a box from another image, mix labels by area | Yun et al., ICCV 2019 | yes | R134–R136 |
| 4 | **Knowledge distillation** into ResNet10S from the ResNet18S ensemble (R62–R64), T = 4, α = 0.7 | Hinton, Vinyals & Dean, 2015 | yes | R137–R139 |
| 5 | **Label-issue check** with confident learning on the out-of-fold predictions of Notebook 09 (ResNet18S, 6 repeats) | Northcutt, Jiang & Chuang, JAIR 2021 | no | – |

Caveat for CutMix: a pasted box can carry a hot cell into a normal module while the label is mixed only by area; this is why CutMix is
applied with p = 0.5 and judged by the results, not assumed to help.

**Comparison:** each fix vs ResNet18S `base` (R62–R64) and ResNet10S (R65–R67), as single models (mean of 3 seeds) and as 3-seed ensembles + TTA,
plus McNemar (seed 42) vs R62. A fix that looks better here would still need the Notebook 09 cross-validation before replacing the main model.

**Needs:** Notebook 08 checkpoints (`checkpoints/R62–R64_ResNet18S_base.pt`), `test_probs/nb08_*`, and Notebook 09 `cv_probs/`.
About 25–35 min on the RTX 4070 Ti. **Close `experiment_log.xlsx` before running.** Finished runs in `results_10.csv` are skipped if you Run All again.

## 0. Imports, device, data

In [ ]:
import time, datetime, platform, json, sys, hashlib
from pathlib import Path
from copy import copy
import numpy as np
import pandas as pd
from PIL import Image
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
import torchvision
from torchvision.models import resnet18, ResNet18_Weights
from sklearn.metrics import (f1_score, accuracy_score, confusion_matrix, ConfusionMatrixDisplay,
                             precision_recall_fscore_support)
from openpyxl import load_workbook

NOTEBOOK = "10_overfitting_fixes.ipynb"
REL = Path("2020-02-14_InfraredSolarModules/InfraredSolarModules")
CANDIDATES = [Path("../archive"), Path("../data/archive"), Path("archive"), Path("../../archive")]
DATA_DIR = next((r / REL for r in CANDIDATES if (r / REL / "images").exists()), None)
if DATA_DIR is None:
    raise FileNotFoundError(f"Data folder not found. This notebook runs in {Path.cwd()}\nLooked in:\n  "
                            + "\n  ".join(str((r / REL).resolve()) for r in CANDIDATES))

if torch.cuda.is_available():
    device = torch.device("cuda"); HARDWARE = torch.cuda.get_device_name(0)
    torch.backends.cudnn.benchmark = True
elif torch.backends.mps.is_available():
    device = torch.device("mps"); HARDWARE = "Apple GPU (mps)"
else:
    device = torch.device("cpu"); HARDWARE = platform.processor() or "CPU"

for d in ("figures", "checkpoints", "runs", "test_probs"):
    Path(d).mkdir(exist_ok=True)
print("PyTorch", torch.__version__, "| torchvision", torchvision.__version__)
print("device:", device, "|", HARDWARE, "| CUDA available:", torch.cuda.is_available(), "| CUDA", torch.version.cuda)
print("data:", DATA_DIR.resolve())

## 1. Split, groups and images (identical to Notebook 05, but **no resize**)

In [ ]:
split = pd.read_csv("splits.csv")
SPLIT_MD5 = hashlib.md5(Path("splits.csv").read_bytes()).hexdigest()
GROUP = {"Cell": "Cell-level", "Cell-Multi": "Cell-level", "Cracking": "Cell-level",
         "Hot-Spot": "Cell-level", "Hot-Spot-Multi": "Cell-level",
         "Diode": "Substring-level", "Diode-Multi": "Substring-level",
         "Offline-Module": "Module-level",
         "Shadowing": "External", "Vegetation": "External", "Soiling": "External",
         "No-Anomaly": "No-Anomaly"}
CLASSES = sorted(split.label.unique())
c2i = {c: i for i, c in enumerate(CLASSES)}
Y = split.label.map(c2i).values
PARTS = {s: np.where(split.split.values == s)[0] for s in ("train", "val", "test")}

t0 = time.time()
imgs = np.stack([np.asarray(Image.open(DATA_DIR / p), dtype=np.float32) / 255.0 for p in split.path])
med = np.median(imgs, axis=(1, 2), keepdims=True)
std = imgs.std(axis=(1, 2), keepdims=True) + 1e-6
Z = torch.from_numpy((imgs - med) / std).unsqueeze(1)          # (20000, 1, 40, 24): this is ALL the preprocessing
print("images:", tuple(Z.shape), f"loaded in {time.time() - t0:.1f}s | splits.csv MD5 {SPLIT_MD5}")

In [ ]:
class IRModules(Dataset):
    # training only: flips, shift +-2 px (replicate padding, crop back to 40x24), optional Gaussian noise
    def __init__(self, idx, augment=False, max_shift=2, noise_std=0.0):
        self.idx = torch.as_tensor(idx); self.y = torch.as_tensor(Y[idx]).long()
        self.augment, self.s, self.noise = augment, max_shift, noise_std
    def __len__(self):
        return len(self.y)
    def __getitem__(self, i):
        x = Z[self.idx[i]]
        if self.augment:
            if torch.rand(1) < 0.5: x = torch.flip(x, dims=[2])
            if torch.rand(1) < 0.5: x = torch.flip(x, dims=[1])
            s = self.s
            if s > 0:
                x = F.pad(x.unsqueeze(0), (s, s, s, s), mode="replicate").squeeze(0)
                dy, dx = torch.randint(0, 2 * s + 1, (2,)).tolist()
                x = x[:, dy:dy + 40, dx:dx + 24]
            if self.noise > 0:
                x = x + self.noise * torch.randn_like(x)
        return x, self.y[i]

def make_loaders(noise_std=0.0, batch=128, seed=42):
    tr = PARTS["train"]
    counts = np.bincount(Y[tr], minlength=len(CLASSES))
    w = counts[Y[tr]] ** -0.5                                         # sampler p ~ 1/sqrt(n), as R10
    g = torch.Generator().manual_seed(seed)
    sampler = WeightedRandomSampler(torch.as_tensor(w, dtype=torch.double), num_samples=len(tr),
                                    replacement=True, generator=g)
    train_dl = DataLoader(IRModules(tr, augment=True, noise_std=noise_std), batch_size=batch, sampler=sampler)
    # fixed 3,000 training images, no augmentation, natural class mix: measures the training loss on the
    # same terms as the validation loss, so the train/validation gap is a fair overfitting measure
    rng = np.random.default_rng(0)
    tr_eval = DataLoader(IRModules(rng.choice(tr, 3000, replace=False)), batch_size=512)
    val_dl = DataLoader(IRModules(PARTS["val"]), batch_size=512)
    test_dl = DataLoader(IRModules(PARTS["test"]), batch_size=512)
    return train_dl, tr_eval, val_dl, test_dl, counts

## 2. The six architectures
All of them take the 40 × 24 × 1 image directly. The comments give the size of the feature map after each stage.

In [ ]:
class ResNet18S(nn.Module):
    # ResNet-18 adapted to small images (He et al. 2016, CIFAR-10 setting): conv1 3x3 stride 1, no max-pool
    # 40x24 -> layer1 40x24 -> layer2 20x12 -> layer3 10x6 -> layer4 5x3 -> global avg-pool -> fc
    def __init__(self, n_classes, pretrained=True, dropout=0.0):
        super().__init__()
        net = resnet18(weights=ResNet18_Weights.IMAGENET1K_V1 if pretrained else None)
        w7 = net.conv1.weight.data.clone()                             # (64, 3, 7, 7) pretrained
        net.conv1 = nn.Conv2d(1, 64, kernel_size=3, stride=1, padding=1, bias=False)
        if pretrained:                                                 # centre 3x3 of each 7x7 filter, RGB summed
            net.conv1.weight.data.copy_(w7[:, :, 2:5, 2:5].sum(dim=1, keepdim=True))
        net.maxpool = nn.Identity()
        net.fc = nn.Sequential(nn.Dropout(dropout), nn.Linear(512, n_classes))
        self.net = net
    def forward(self, x):
        return self.net(x)

class ResNet10S(ResNet18S):
    # same as ResNet18S but only the FIRST basic block of each stage is kept (8 conv layers + fc instead of 16 + fc)
    def __init__(self, n_classes, pretrained=True, dropout=0.0):
        super().__init__(n_classes, pretrained, dropout)
        for name in ("layer1", "layer2", "layer3", "layer4"):
            setattr(self.net, name, nn.Sequential(getattr(self.net, name)[0]))

class WideBasic(nn.Module):
    # pre-activation block of Wide ResNet: BN-ReLU-conv3x3 -> (dropout) -> BN-ReLU-conv3x3, plus shortcut
    def __init__(self, c_in, c_out, stride, dropout):
        super().__init__()
        self.bn1, self.conv1 = nn.BatchNorm2d(c_in), nn.Conv2d(c_in, c_out, 3, stride, 1, bias=False)
        self.bn2, self.conv2 = nn.BatchNorm2d(c_out), nn.Conv2d(c_out, c_out, 3, 1, 1, bias=False)
        self.drop = nn.Dropout(dropout)
        self.short = None if (stride == 1 and c_in == c_out) else nn.Conv2d(c_in, c_out, 1, stride, bias=False)
    def forward(self, x):
        o = F.relu(self.bn1(x))
        sc = x if self.short is None else self.short(o)
        o = self.conv1(o)
        o = self.conv2(self.drop(F.relu(self.bn2(o))))
        return o + sc

class WRN(nn.Module):
    # Wide ResNet (Zagoruyko & Komodakis 2016). WRN-16-4: 3 groups x 2 blocks, widths 64/128/256
    # 40x24 -> group1 40x24 -> group2 20x12 -> group3 10x6 -> global avg-pool -> fc
    def __init__(self, n_classes, depth=16, k=4, dropout=0.0):
        super().__init__()
        n = (depth - 4) // 6
        widths = [16, 16 * k, 32 * k, 64 * k]
        self.conv = nn.Conv2d(1, widths[0], 3, 1, 1, bias=False)
        blocks, c_in = [], widths[0]
        for g, stride in enumerate([1, 2, 2]):
            for i in range(n):
                blocks.append(WideBasic(c_in, widths[g + 1], stride if i == 0 else 1, dropout))
                c_in = widths[g + 1]
        self.blocks = nn.Sequential(*blocks)
        self.bn = nn.BatchNorm2d(c_in)
        self.fc = nn.Linear(c_in, n_classes)
    def forward(self, x):
        o = F.relu(self.bn(self.blocks(self.conv(x))))
        return self.fc(F.adaptive_avg_pool2d(o, 1).flatten(1))

def conv_bn(c_in, c_out, k=3):
    return nn.Sequential(nn.Conv2d(c_in, c_out, k, padding=k // 2, bias=False), nn.BatchNorm2d(c_out), nn.ReLU())

class MSCNN(nn.Module):
    # multi-scale CNN, idea of Korkmaz & Acikgoz 2022 (three branches with different kernel sizes);
    # simplified re-implementation. branches 3x3 / 5x5 / 7x7 at 40x24 -> concat 96 ch
    # -> 128 ch, pool 20x12 -> 128 ch, pool 10x6 -> 256 ch -> global avg-pool -> fc
    def __init__(self, n_classes, dropout=0.0):
        super().__init__()
        self.branches = nn.ModuleList([nn.Sequential(conv_bn(1, 32, k), conv_bn(32, 32, k)) for k in (3, 5, 7)])
        self.body = nn.Sequential(conv_bn(96, 128), nn.MaxPool2d(2), conv_bn(128, 128), nn.MaxPool2d(2),
                                  conv_bn(128, 256))
        self.head = nn.Sequential(nn.Dropout(dropout), nn.Linear(256, n_classes))
    def forward(self, x):
        o = torch.cat([b(x) for b in self.branches], dim=1)
        o = self.body(o)
        return self.head(F.adaptive_avg_pool2d(o, 1).flatten(1))

class CifarBasic(nn.Module):
    # basic residual block of the CIFAR ResNets (He et al. 2016)
    def __init__(self, c_in, c_out, stride):
        super().__init__()
        self.c1, self.b1 = nn.Conv2d(c_in, c_out, 3, stride, 1, bias=False), nn.BatchNorm2d(c_out)
        self.c2, self.b2 = nn.Conv2d(c_out, c_out, 3, 1, 1, bias=False), nn.BatchNorm2d(c_out)
        self.short = (nn.Identity() if (stride == 1 and c_in == c_out) else
                      nn.Sequential(nn.Conv2d(c_in, c_out, 1, stride, bias=False), nn.BatchNorm2d(c_out)))
    def forward(self, x):
        o = F.relu(self.b1(self.c1(x)))
        return F.relu(self.b2(self.c2(o)) + self.short(x))

class ResNetCifar(nn.Module):
    # ResNet-20 for small images (He et al. 2016, CIFAR-10): 3 stages x 3 blocks, widths 16/32/64
    # 40x24 -> stage1 40x24 -> stage2 20x12 -> stage3 10x6 -> global avg-pool -> fc
    def __init__(self, n_classes, depth=20, dropout=0.0):
        super().__init__()
        n = (depth - 2) // 6
        self.stem = nn.Sequential(nn.Conv2d(1, 16, 3, 1, 1, bias=False), nn.BatchNorm2d(16), nn.ReLU())
        layers, c_in = [], 16
        for c_out, stride in [(16, 1), (32, 2), (64, 2)]:
            for i in range(n):
                layers.append(CifarBasic(c_in, c_out, stride if i == 0 else 1)); c_in = c_out
        self.layers = nn.Sequential(*layers)
        self.head = nn.Sequential(nn.Dropout(dropout), nn.Linear(64, n_classes))
    def forward(self, x):
        o = self.layers(self.stem(x))
        return self.head(F.adaptive_avg_pool2d(o, 1).flatten(1))

class DenseLayer(nn.Module):
    # DenseNet-BC bottleneck layer: BN-ReLU-conv1x1 (4k) -> BN-ReLU-conv3x3 (k), output concatenated to its input
    def __init__(self, c_in, k):
        super().__init__()
        self.f = nn.Sequential(nn.BatchNorm2d(c_in), nn.ReLU(), nn.Conv2d(c_in, 4 * k, 1, bias=False),
                               nn.BatchNorm2d(4 * k), nn.ReLU(), nn.Conv2d(4 * k, k, 3, 1, 1, bias=False))
    def forward(self, x):
        return torch.cat([x, self.f(x)], dim=1)

class DenseNetBC(nn.Module):
    # DenseNet-BC for small images (Huang et al. 2017), L = 100, k = 12, compression 0.5
    # 40x24 -> block1 40x24 -> transition 20x12 -> block2 -> transition 10x6 -> block3 -> global avg-pool -> fc
    def __init__(self, n_classes, depth=100, k=12, dropout=0.0):
        super().__init__()
        n = (depth - 4) // 6
        c = 2 * k
        mods = [nn.Conv2d(1, c, 3, 1, 1, bias=False)]
        for b in range(3):
            for _ in range(n):
                mods.append(DenseLayer(c, k)); c += k
            if b < 2:
                c_out = c // 2
                mods += [nn.BatchNorm2d(c), nn.ReLU(), nn.Conv2d(c, c_out, 1, bias=False), nn.AvgPool2d(2)]
                c = c_out
        mods += [nn.BatchNorm2d(c), nn.ReLU()]
        self.features = nn.Sequential(*mods)
        self.head = nn.Sequential(nn.Dropout(dropout), nn.Linear(c, n_classes))
    def forward(self, x):
        return self.head(F.adaptive_avg_pool2d(self.features(x), 1).flatten(1))

def build_model(name, n_classes, dropout):
    return {"ResNet18S": lambda: ResNet18S(n_classes, pretrained=True, dropout=dropout),
            "ResNet10S": lambda: ResNet10S(n_classes, pretrained=True, dropout=dropout),
            "ResNet18S_scratch": lambda: ResNet18S(n_classes, pretrained=False, dropout=dropout),
            "WRN16_4": lambda: WRN(n_classes, 16, 4, dropout=dropout),
            "MSCNN": lambda: MSCNN(n_classes, dropout=dropout),
            "ResNet20": lambda: ResNetCifar(n_classes, 20, dropout=dropout),
            "DenseNetBC100": lambda: DenseNetBC(n_classes, 100, 12, dropout=dropout)}[name]()

MODEL_INFO = {
    "ResNet10S": "ResNet-10 = ResNet18S with 1 basic block per stage (first block of each ImageNet-pretrained stage kept), small-image stem "
                 "(conv1 3x3 s1, no max-pool), 1-channel 40x24 input, all layers fine-tuned",
    "ResNet18S": "ResNet-18, small-image stem (conv1 3x3 s1, no max-pool; He et al. 2016 CIFAR setting), 1-channel 40x24 input, "
                 "layer1-4 ImageNet-pretrained (IMAGENET1K_V1), conv1 from centre 3x3 of pretrained 7x7 (RGB summed), all layers fine-tuned",
    "ResNet18S_scratch": "ResNet-18, small-image stem (conv1 3x3 s1, no max-pool), 1-channel 40x24 input, random initialisation",
    "WRN16_4": "Wide ResNet-16-4 (Zagoruyko & Komodakis 2016), pre-activation blocks, widths 64/128/256, 1-channel 40x24 input, random initialisation",
    "ResNet20": "ResNet-20 for small images (He et al. 2016, CIFAR-10): 3 stages x 3 basic blocks, widths 16/32/64, 1-channel 40x24 input, random initialisation",
    "DenseNetBC100": "DenseNet-BC L=100 k=12 for small images (Huang et al. 2017), compression 0.5, 1-channel 40x24 input, random initialisation",
    "MSCNN": "Multi-scale CNN (idea of Korkmaz & Acikgoz 2022, simplified): branches 3x3/5x5/7x7 (32 ch each) -> 128 -> pool -> 128 -> pool -> 256 -> GAP -> fc, random initialisation",
}
LR = {"ResNet18S": 3e-4, "ResNet10S": 3e-4, "ResNet18S_scratch": 1e-3, "WRN16_4": 1e-3, "MSCNN": 1e-3, "ResNet20": 1e-3, "DenseNetBC100": 1e-3}       # pretrained: smaller steps
EPOCHS = {"ResNet18S": 30, "ResNet10S": 30, "ResNet18S_scratch": 40, "WRN16_4": 40, "MSCNN": 40, "ResNet20": 40, "DenseNetBC100": 40}           # from scratch: more epochs
RECIPES = {
    "base": dict(label_smoothing=0.0, dropout=0.0, weight_decay=1e-4, noise_std=0.0),
    "reg":  dict(label_smoothing=0.1, dropout=0.3, weight_decay=5e-4, noise_std=0.05),
    "reg_noLS": dict(label_smoothing=0.0, dropout=0.3, weight_decay=5e-4, noise_std=0.05),
}
USED = ["ResNet18S", "ResNet10S"]

# self-test: shapes, parameters and one training step per model (nothing is logged)
for name in USED:
    m = build_model(name, 12, 0.3).to(device)
    n_par = sum(p.numel() for p in m.parameters() if p.requires_grad)
    x = Z[:8].to(device); y = torch.as_tensor(Y[:8]).to(device)
    opt = torch.optim.AdamW(m.parameters(), lr=1e-4)
    loss = F.cross_entropy(m.train()(x), y); loss.backward(); opt.step()
    with torch.no_grad():
        out = m.eval()(x)
    print(f"{name:18s} parameters = {n_par:>10,}   input {tuple(x.shape)} -> output {tuple(out.shape)}   one step OK")
del m, opt

## 3. One function that trains, tests and logs a run
Same as Notebook 05, plus:
* the **training-subset loss** (3,000 fixed training images, no augmentation) is computed every epoch with **plain cross-entropy**, like the validation loss (label smoothing raises the training loss on purpose, so the smoothed loss is not used for the comparison);
* new overfitting numbers per run: lowest validation loss and its epoch, validation loss at the last epoch, **rise** = last − lowest (0 means no rise), and **gap** = validation loss − training-subset loss at the last epoch.

In [ ]:
def environment():
    return {"python": sys.version.split()[0], "pytorch": torch.__version__, "torchvision": torchvision.__version__,
            "cuda": torch.version.cuda, "cudnn": torch.backends.cudnn.version() if torch.backends.cudnn.is_available() else None,
            "device": str(device), "hardware": HARDWARE, "os": platform.platform(),
            "data_dir": str(DATA_DIR.resolve()), "splits_md5": SPLIT_MD5}

def logged_ids():
    ids = set()
    for f in ("results_05.csv", "results_06.csv", "results_08.csv", "results_09.csv", "results_10.csv"):
        if Path(f).exists():
            ids |= set(pd.read_csv(f).run_id.astype(str))
    if Path("experiment_log.xlsx").exists():
        ws = load_workbook("experiment_log.xlsx", read_only=True)["Runs"]
        ids |= {str(r[0]) for r in ws.iter_rows(min_row=2, values_only=True) if r and r[0]}
    return ids

def next_free_id(run_id):
    ids = logged_ids()
    if run_id not in ids:
        return run_id
    k = 1
    while f"{run_id}-rerun{k}" in ids:
        k += 1
    return f"{run_id}-rerun{k}"

def evaluate(model, dl):
    model.eval(); ys, ps, loss_sum = [], [], 0.0
    with torch.no_grad():
        for x, y in dl:
            out = model(x.to(device))
            loss_sum += F.cross_entropy(out, y.to(device), reduction="sum").item()      # plain CE, no smoothing
            ps.append(out.argmax(1).cpu()); ys.append(y)
    yt, pt = torch.cat(ys).numpy(), torch.cat(ps).numpy()
    return yt, pt, loss_sum / len(yt)

K_OF = {"No-Anomaly": 0, "Diode": 1, "Diode-Multi": 2, "Offline-Module": 3}   # bypassed substrings per module (others 0)
K_ARR = np.array([K_OF.get(c, 0) for c in CLASSES])

def predict_probs(model, idx, tta=False):
    # softmax probabilities on images idx; tta = mean over original, H flip, V flip, H+V flip
    model.eval(); out = []
    with torch.no_grad():
        for b in range(0, len(idx), 512):
            x = Z[idx[b:b + 512]].to(device)
            views = [x] + ([torch.flip(x, [3]), torch.flip(x, [2]), torch.flip(x, [2, 3])] if tta else [])
            out.append(torch.stack([F.softmax(model(v), 1) for v in views]).mean(0).cpu())
    return torch.cat(out).numpy()

def ece_score(probs, yt, bins=15):
    conf, pred = probs.max(1), probs.argmax(1); e = 0.0
    edges = np.linspace(0, 1, bins + 1)
    for lo, hi in zip(edges[:-1], edges[1:]):
        m = (conf > lo) & (conf <= hi)
        if m.any():
            e += m.mean() * abs((pred[m] == yt[m]).mean() - conf[m].mean())
    return float(e)

def metrics_from_probs(probs, yt):
    names = np.array(CLASSES); pt = probs.argmax(1)
    kt, kp = K_ARR[yt], K_ARR[pt]
    return dict(acc=accuracy_score(yt, pt), f1=f1_score(yt, pt, average="macro"),
                f1_5=f1_score([GROUP[c] for c in names[yt]], [GROUP[c] for c in names[pt]], average="macro"),
                fault_acc=float(np.mean((names[yt] == "No-Anomaly") == (names[pt] == "No-Anomaly"))),
                nll=float(-np.mean(np.log(np.clip(probs[np.arange(len(yt)), yt], 1e-12, 1)))),
                ece=ece_score(probs, yt), k_acc=float(np.mean(kt == kp)),
                ploss_err_pct=float(100 * (kp.sum() - kt.sum()) / kt.sum()))

def _replace_rows(ws, run_id, rows):
    for r in range(ws.max_row, 1, -1):
        if ws.cell(r, 1).value == run_id:
            ws.delete_rows(r)
    style_row = ws.max_row
    for values in rows:
        ws.append(values); r = ws.max_row
        for c in range(1, len(values) + 1):
            src, dst = ws.cell(style_row, c), ws.cell(r, c)
            if src.has_style:
                dst.font, dst.border, dst.alignment = copy(src.font), copy(src.border), copy(src.alignment)
                dst.number_format = src.number_format

def log_run(res, params, note=""):
    path = Path("experiment_log.xlsx")
    if not path.exists():
        print("experiment_log.xlsx not found: results not logged"); return
    wb = load_workbook(path)
    _replace_rows(wb["Runs"], res["run_id"], [[
        res["run_id"], datetime.date.today().isoformat(), NOTEBOOK, res["model"] + f" [{res['recipe']}]",
        "Raptor Maps, 12 classes", "splits.csv (70/15/15, seed 42)", "Raw pixels, per-image z-score, native 40x24 (no resize)",
        round(res["val_acc"], 3), round(res["val_f1"], 3), round(res["test_acc"], 3), round(res["test_f1"], 3),
        round(res["fault_acc"], 3), round(res["train_time"], 1), HARDWARE,
        (note + "; " if note else "") + f"best epoch {res['best_epoch']}; min val loss {res['val_loss_min']:.3f} @ ep {res['val_loss_min_epoch']}; "
        f"early-stopping ckpt: test F1 {res['es_f1']:.3f}, NLL {res['es_nll']:.3f}"]])
    _replace_rows(wb["Parameters"], res["run_id"], [[res["run_id"], "Run", k, v] for k, v in params.items()])
    wb.save(path)
    print(f"{res['run_id']} logged in experiment_log.xlsx")

def run_experiment(run_id, model_name, recipe, seed=42, batch=128, max_shift=2, note=""):
    rc = RECIPES[recipe]; lr, epochs = LR[model_name], EPOCHS[model_name]
    torch.manual_seed(seed); np.random.seed(seed)
    rerun_of = None
    free_id = next_free_id(run_id)
    if free_id != run_id:
        rerun_of, run_id = run_id, free_id
        note = f"Re-run of {rerun_of} with identical settings (original kept)" + (f"; {note}" if note else "")
    started = datetime.datetime.now()
    run_dir = Path("runs") / f"{run_id}_{started:%Y%m%d-%H%M%S}"
    run_dir.mkdir(parents=True, exist_ok=True)
    logf = open(run_dir / "train.log", "w", encoding="utf-8")
    def log(msg=""):
        line = f"[{datetime.datetime.now():%H:%M:%S}] {msg}"
        print(line); logf.write(line + "\n"); logf.flush()

    train_dl, tr_eval, val_dl, test_dl, counts = make_loaders(rc["noise_std"], batch, seed)
    train_dl.dataset.s = max_shift
    model = build_model(model_name, len(CLASSES), rc["dropout"]).to(device)
    n_par = sum(p.numel() for p in model.parameters() if p.requires_grad)
    criterion = nn.CrossEntropyLoss(label_smoothing=rc["label_smoothing"])
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=rc["weight_decay"])
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)
    ckpt = Path("checkpoints") / f"{run_id}_{model_name}_{recipe}.pt"                 # best validation macro-F1
    ckpt_es = Path("checkpoints") / f"{run_id}_{model_name}_{recipe}_minloss.pt"     # lowest validation loss (early stopping)

    log(f"===== {run_id}: {model_name}, recipe = {recipe}, seed {seed}, input 1x40x24 (no resize) =====")
    log(f"folder {run_dir} | {n_par:,} parameters | lr {lr:g} | epochs {epochs} | batch {batch} | {rc} | {HARDWARE}")
    hist, best, best_ep, best_acc, t0 = [], -1.0, 0, 0.0, time.time()
    best_vl, es_ep = float("inf"), 0
    for ep in range(1, epochs + 1):
        model.train(); total, correct, seen = 0.0, 0, 0
        lr_now = optimizer.param_groups[0]["lr"]
        for x, y in train_dl:
            x, y = x.to(device), y.to(device)
            optimizer.zero_grad()
            out = model(x); loss = criterion(out, y)
            loss.backward(); optimizer.step()
            total += loss.item() * len(y); correct += (out.argmax(1) == y).sum().item(); seen += len(y)
        scheduler.step()
        ye, pe, tr_loss = evaluate(model, tr_eval)
        yv, pv, val_loss = evaluate(model, val_dl)
        f1v, accv = f1_score(yv, pv, average="macro"), accuracy_score(yv, pv)
        hist.append({"epoch": ep, "lr": lr_now, "train_loss_batches": total / seen, "train_acc_batches": correct / seen,
                     "train_eval_loss": tr_loss, "train_eval_acc": accuracy_score(ye, pe),
                     "val_loss": val_loss, "val_acc": accv, "val_macroF1": f1v, "seconds": round(time.time() - t0, 1)})
        if f1v > best:
            best, best_ep, best_acc = f1v, ep, accv
            torch.save(model.state_dict(), ckpt)
        if val_loss < best_vl:
            best_vl, es_ep = val_loss, ep
            torch.save(model.state_dict(), ckpt_es)
        log(f"epoch {ep:2d} | lr {lr_now:.2e} | train-subset loss {tr_loss:.3f} | val loss {val_loss:.3f} acc {accv:.3f} "
            f"macro-F1 {f1v:.3f}{' *' if ep == best_ep else ''}{' [min loss]' if ep == es_ep else ''} | {time.time() - t0:.0f}s")
    train_time = time.time() - t0
    h = pd.DataFrame(hist); h.to_csv(run_dir / "history.csv", index=False)

    model.load_state_dict(torch.load(ckpt, map_location=device))
    yt, pt, test_loss = evaluate(model, test_dl)
    names = np.array(CLASSES)
    i_min = int(h.val_loss.idxmin())
    res = dict(run_id=run_id, rerun_of=rerun_of, model=model_name, recipe=recipe, seed=seed,
               val_acc=best_acc, val_f1=best, best_epoch=best_ep,
               test_acc=accuracy_score(yt, pt), test_f1=f1_score(yt, pt, average="macro"),
               test_f1_as_5=f1_score([GROUP[c] for c in names[yt]], [GROUP[c] for c in names[pt]], average="macro"),
               fault_acc=float(np.mean((names[yt] == "No-Anomaly") == (names[pt] == "No-Anomaly"))),
               test_loss=test_loss,
               val_loss_min=float(h.val_loss.min()), val_loss_min_epoch=int(h.epoch[i_min]),
               val_loss_last=float(h.val_loss.iloc[-1]), val_loss_at_best=float(h.val_loss[h.epoch == best_ep].iloc[0]),
               val_loss_rise=float(h.val_loss.iloc[-1] - h.val_loss.min()),
               gap_last=float(h.val_loss.iloc[-1] - h.train_eval_loss.iloc[-1]),
               train_time=train_time, params=n_par)
    te = PARTS["test"]
    for sel, path in (("f1sel", ckpt), ("es", ckpt_es)):
        model.load_state_dict(torch.load(path, map_location=device))
        pr = predict_probs(model, te); pr_tta = predict_probs(model, te, tta=True)
        np.save(Path("test_probs") / f"nb08_{run_id}_{sel}.npy", pr)
        np.save(Path("test_probs") / f"nb08_{run_id}_{sel}_tta.npy", pr_tta)
        for k, v in metrics_from_probs(pr, yt).items():
            res[f"{sel}_{k}"] = v
    res["es_epoch"] = es_ep
    model.load_state_dict(torch.load(ckpt, map_location=device))
    p, r, f, s = precision_recall_fscore_support(yt, pt, labels=range(len(CLASSES)), zero_division=0)
    per_class = pd.DataFrame({"class": CLASSES, "precision": p, "recall": r, "f1": f, "test_images": s}).round(3)
    per_class.to_csv(run_dir / "test_per_class.csv", index=False)
    cm_counts = confusion_matrix(yt, pt, labels=range(len(CLASSES)))
    pd.DataFrame(cm_counts, index=[f"true {c}" for c in CLASSES], columns=[f"pred {c}" for c in CLASSES]).to_csv(run_dir / "confusion_matrix.csv")
    pd.DataFrame({"path": split.path.values[PARTS["test"]], "true": names[yt], "pred": names[pt]}).to_csv(run_dir / "test_predictions.csv", index=False)

    log(f"best epoch {best_ep} | val macro-F1 {best:.3f} | lowest val loss {res['val_loss_min']:.3f} @ epoch {res['val_loss_min_epoch']} "
        f"| last val loss {res['val_loss_last']:.3f} (rise {res['val_loss_rise']:.3f}, gap {res['gap_last']:.3f}) | {train_time:.0f}s")
    log(f"TEST accuracy {res['test_acc']:.3f} | macro-F1 {res['test_f1']:.3f} | as 5 groups {res['test_f1_as_5']:.3f} | fault/no-fault {res['fault_acc']:.3f}")
    log(f"EARLY STOPPING (min val loss, epoch {es_ep}): TEST accuracy {res['es_acc']:.3f} | macro-F1 {res['es_f1']:.3f} | "
        f"5 groups {res['es_f1_5']:.3f} | NLL {res['es_nll']:.3f} (best-F1 checkpoint {res['f1sel_nll']:.3f}) | "
        f"power-loss error {res['es_ploss_err_pct']:+.1f}% (best-F1 checkpoint {res['f1sel_ploss_err_pct']:+.1f}%)")

    fig, ax = plt.subplots(1, 3, figsize=(16, 4.5), gridspec_kw={"width_ratios": [1, 1, 1.4]})
    ax[0].plot(h.epoch, h.train_eval_loss, color="#2E5E8C", lw=2, label="train subset")
    ax[0].plot(h.epoch, h.val_loss, color="#8A1F2B", lw=2, label="validation")
    ax[0].axvline(res["val_loss_min_epoch"], color="#888", ls=":", lw=1)
    ax[0].set_title("Loss (plain cross-entropy)"); ax[0].set_xlabel("epoch"); ax[0].legend(frameon=False)
    ax[1].plot(h.epoch, h.val_macroF1, color="#8A1F2B", lw=2); ax[1].axvline(best_ep, color="#888", ls="--", lw=1)
    ax[1].set_title("Validation macro-F1"); ax[1].set_xlabel("epoch")
    cm = confusion_matrix(yt, pt, labels=range(len(CLASSES)), normalize="true")
    ConfusionMatrixDisplay(cm, display_labels=CLASSES).plot(ax=ax[2], cmap="Reds", values_format=".2f", colorbar=False)
    ax[2].tick_params(axis="x", rotation=60); ax[2].set_title(f"{run_id}: test confusion matrix")
    for a in ax[:2]: a.grid(alpha=0.25)
    plt.tight_layout(); plt.savefig(run_dir / "curves_and_cm.png", dpi=200)
    plt.savefig(f"figures/{run_id}_{model_name}_{recipe}.png", dpi=200); plt.close(fig)

    config = {"run_id": run_id, "rerun_of": rerun_of, "note": note, "started": started.isoformat(timespec="seconds"),
              "notebook": NOTEBOOK, "run_folder": str(run_dir), "checkpoint": str(ckpt),
              "data": {"dataset": "Raptor Maps InfraredSolarModules (2020-02-14)", "split_file": "splits.csv", "classes": CLASSES,
                       "train_images_per_class": dict(zip(CLASSES, counts.tolist()))},
              "input": "1 x 40 x 24, per-image z = (I - median) / (std + 1e-6), NO resize, NO channel repeat",
              "augmentation": {"hflip_p": 0.5, "vflip_p": 0.5, "random_shift_px": max_shift, "shift_padding": "replicate",
                               "gaussian_noise_std": rc["noise_std"], "applied_to": "training set only"},
              "balancing": "WeightedRandomSampler, p = 1/sqrt(n_class), 14,000 draws per epoch, replacement",
              "model": {"name": model_name, "description": MODEL_INFO[model_name], "trainable_parameters": n_par},
              "training": {"recipe": recipe, "loss": f"CrossEntropyLoss(label_smoothing={rc['label_smoothing']})",
                           "dropout": rc["dropout"], "optimizer": "AdamW", "lr": lr, "weight_decay": rc["weight_decay"],
                           "lr_schedule": f"CosineAnnealingLR(T_max={epochs})", "epochs": epochs, "batch_train": batch,
                           "batch_eval": 512, "seed": seed, "model_selection": "two checkpoints: best validation macro-F1 (main) and lowest validation loss (early stopping)",
                           "overfit_monitor": "plain CE on 3,000 fixed training images (no augmentation) and on validation"},
              "environment": environment(),
              "results": {k: (round(float(v), 4) if isinstance(v, (float, np.floating)) else v) for k, v in res.items()}}
    config["results"]["finished"] = datetime.datetime.now().isoformat(timespec="seconds")
    (run_dir / "config.json").write_text(json.dumps(config, indent=2, ensure_ascii=False, default=str), encoding="utf-8")

    params = {
        "Run folder": str(run_dir), "Classes": f"12 ({', '.join(CLASSES)})",
        "Input": "1 x 40 x 24 native, per-image z-score; no resize, no channel repeat",
        "Balancing": "WeightedRandomSampler, p ∝ 1/sqrt(n_class), 14,000 draws per epoch, replacement",
        "Augmentation": f"H flip p=0.5, V flip p=0.5, shift ±{max_shift} px (replicate), Gaussian noise σ={rc['noise_std']}; training only",
        "Model": MODEL_INFO[model_name], "Trainable parameters": f"{n_par:,}",
        "Recipe": recipe, "Loss": f"Cross-entropy, label smoothing {rc['label_smoothing']}", "Dropout": str(rc["dropout"]),
        "Optimizer": f"AdamW, lr = {lr:g}, weight_decay = {rc['weight_decay']:g}",
        "LR schedule": f"CosineAnnealingLR, T_max = {epochs}", "Epochs / batch": f"{epochs} / {batch} (eval batch 512)",
        "Model selection": f"best validation macro-F1 → epoch {best_ep}; early stopping (lowest validation loss) → epoch {es_ep}",
        "Early-stopping checkpoint, test": (f"acc {res['es_acc']:.3f}, macro-F1 {res['es_f1']:.3f}, 5 groups {res['es_f1_5']:.3f}, "
                                            f"NLL {res['es_nll']:.3f}, ECE {res['es_ece']:.3f}, power-loss error {res['es_ploss_err_pct']:+.1f}%"),
        "Overfitting": (f"min val loss {res['val_loss_min']:.3f} @ epoch {res['val_loss_min_epoch']}; last {res['val_loss_last']:.3f}; "
                        f"rise {res['val_loss_rise']:.3f}; gap (val − train subset) {res['gap_last']:.3f}"),
        "Random seed": f"{seed} (torch, numpy, sampler; data split always seed 42)", "splits.csv MD5": SPLIT_MD5,
        "Environment": f"Python {sys.version.split()[0]}, PyTorch {torch.__version__}, CUDA {torch.version.cuda}, {device} ({HARDWARE}), {platform.platform()}",
        "Test macro-F1 scored as 5 groups": f"{res['test_f1_as_5']:.3f}",
        "Test precision / recall / F1 per class": "; ".join(f"{row['class']} {row.precision}/{row.recall}/{row.f1}" for _, row in per_class.iterrows()),
    }
    log_run(res, params, note)
    log(f"saved: {run_dir}"); logf.close()
    return res

RES_COLS = ["run_id", "rerun_of", "model", "recipe", "seed", "val_f1", "best_epoch", "test_acc", "test_f1", "test_f1_as_5", "fault_acc",
            "test_loss", "val_loss_min", "val_loss_min_epoch", "val_loss_at_best", "val_loss_last", "val_loss_rise", "gap_last", "train_time", "params",
            "es_epoch"] + [f"{sel}_{k}" for sel in ("f1sel", "es") for k in ("acc", "f1", "f1_5", "fault_acc", "nll", "ece", "k_acc", "ploss_err_pct")]
def save_result(r):
    new = pd.DataFrame([r]).reindex(columns=RES_COLS)
    old = pd.read_csv("results_08.csv").reindex(columns=RES_COLS) if Path("results_08.csv").exists() else pd.DataFrame(columns=RES_COLS)
    pd.concat([old[~old.run_id.isin(new.run_id)], new], ignore_index=True).to_csv("results_08.csv", index=False)

## 4. Fix 1: Temperature scaling (no training)
The probabilities are divided by a temperature T before softmax: p = softmax(z / T). T is chosen to minimise the validation NLL, so
predictions (and accuracy) do not change; only the confidence does. For the ensemble, T is applied to the log of the averaged probabilities.

In [ ]:
from scipy.optimize import minimize_scalar
def load_model(run_id, name="ResNet18S", recipe="base"):
    m = build_model(name, len(CLASSES), 0.0).to(device)
    m.load_state_dict(torch.load(Path("checkpoints") / f"{run_id}_{name}_{recipe}.pt", map_location=device)); m.eval(); return m
TEACHERS = {r: load_model(r) for r in ("R62", "R63", "R64")}
yv, yt = Y[PARTS["val"]], Y[PARTS["test"]]

def scale(P, T):
    z = np.log(np.clip(P, 1e-12, 1)) / T; z -= z.max(1, keepdims=True); e = np.exp(z); return e / e.sum(1, keepdims=True)
def fit_T(Pv, yv):
    f = lambda lt: -np.mean(np.log(np.clip(scale(Pv, np.exp(lt))[np.arange(len(yv)), yv], 1e-12, 1)))
    return float(np.exp(minimize_scalar(f, bounds=(-3, 3), method="bounded").x))

ts_rows = []
for label, ids in (("ResNet18S single (R62)", ["R62"]), ("ResNet18S ensemble 3 seeds + TTA", ["R62", "R63", "R64"])):
    tta = len(ids) > 1
    Pv = np.mean([predict_probs(TEACHERS[r], PARTS["val"], tta=tta) for r in ids], axis=0)
    Pt = np.mean([np.load(Path("test_probs") / f"nb08_{r}_f1sel{'_tta' if tta else ''}.npy") for r in ids], axis=0)
    T = fit_T(Pv, yv)
    before, after = metrics_from_probs(Pt, yt), metrics_from_probs(scale(Pt, T), yt)
    ts_rows.append(dict(model=label, T=round(T, 3), acc=before["acc"], f1=before["f1"], nll_before=before["nll"], nll_after=after["nll"],
                        ece_before=before["ece"], ece_after=after["ece"]))
ts = pd.DataFrame(ts_rows).round(4); ts.to_csv("results_10_temperature.csv", index=False); print(ts.to_string(index=False))

## 5. Fixes 2–4: one training function with three modes
* `mixup`: λ ~ Beta(0.2, 0.2); x̃ = λx_i + (1 − λ)x_j; loss = λ·CE(y_i) + (1 − λ)·CE(y_j).
* `cutmix`: with p = 0.5, λ ~ Beta(1, 1); paste a box of area (1 − λ) from x_j; λ is re-computed from the exact box area.
* `kd`: student ResNet10S; teacher = mean softmax of R62–R64 at temperature T = 4 on the same augmented batch;
  loss = α·T²·KL(p_teacher^T ‖ p_student^T) + (1 − α)·CE(y), α = 0.7.

Everything else is the `base` recipe (AdamW 3e-4, wd 1e-4, cosine, 30 epochs, batch 128, sampler p ∝ 1/√n, flips + shift);
checkpoint = best validation macro-F1, as for R62–R67. The training-subset and validation losses are plain cross-entropy, as before.

In [ ]:
MIX = dict(mixup_alpha=0.2, cutmix_alpha=1.0, cutmix_p=0.5, kd_T=4.0, kd_alpha=0.7)
COLS10 = ["run_id", "model", "mode", "seed", "best_epoch", "val_f1", "acc", "f1", "f1_5", "fault_acc", "nll", "ece", "k_acc", "ploss_err_pct",
          "tta_acc", "tta_f1", "val_loss_min", "val_loss_last", "val_loss_rise", "gap_last", "train_time", "params"]

def teacher_probs(x, T):
    with torch.no_grad():
        return torch.stack([F.softmax(m(x) / T, 1) for m in TEACHERS.values()]).mean(0)

def run_fix(run_id, model_name, mode, seed, batch=128):
    lr, epochs, wd = LR[model_name], EPOCHS[model_name], RECIPES["base"]["weight_decay"]
    torch.manual_seed(seed); np.random.seed(seed); rng = np.random.default_rng(seed)
    started = datetime.datetime.now()
    run_dir = Path("runs") / f"{run_id}_{started:%Y%m%d-%H%M%S}"; run_dir.mkdir(parents=True, exist_ok=True)
    logf = open(run_dir / "train.log", "w", encoding="utf-8")
    def log(msg=""):
        line = f"[{datetime.datetime.now():%H:%M:%S}] {msg}"; print(line); logf.write(line + "\n"); logf.flush()
    train_dl, tr_eval, val_dl, test_dl, counts = make_loaders(0.0, batch, seed)
    model = build_model(model_name, len(CLASSES), 0.0).to(device)
    n_par = sum(p.numel() for p in model.parameters() if p.requires_grad)
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=wd)
    sch = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs)
    ckpt = Path("checkpoints") / f"{run_id}_{model_name}_{mode}.pt"
    log(f"===== {run_id}: {model_name}, mode = {mode}, seed {seed}, {MIX} | {n_par:,} parameters | {HARDWARE} =====")
    hist, best, best_ep, t0 = [], -1.0, 0, time.time()
    for ep in range(1, epochs + 1):
        model.train()
        for x, y in train_dl:
            x, y = x.to(device), y.to(device)
            if mode == "mixup":
                lam = float(rng.beta(MIX["mixup_alpha"], MIX["mixup_alpha"])); perm = torch.randperm(len(y), device=device)
                out = model(lam * x + (1 - lam) * x[perm])
                loss = lam * F.cross_entropy(out, y) + (1 - lam) * F.cross_entropy(out, y[perm])
            elif mode == "cutmix" and rng.random() < MIX["cutmix_p"]:
                lam = float(rng.beta(MIX["cutmix_alpha"], MIX["cutmix_alpha"])); perm = torch.randperm(len(y), device=device)
                H, W = x.shape[2], x.shape[3]; r = np.sqrt(1 - lam)
                h, w = int(H * r), int(W * r); cy, cx = rng.integers(H), rng.integers(W)
                y1, y2, x1, x2 = max(cy - h // 2, 0), min(cy + h // 2, H), max(cx - w // 2, 0), min(cx + w // 2, W)
                x = x.clone(); x[:, :, y1:y2, x1:x2] = x[perm, :, y1:y2, x1:x2]
                lam = 1 - (y2 - y1) * (x2 - x1) / (H * W)
                out = model(x); loss = lam * F.cross_entropy(out, y) + (1 - lam) * F.cross_entropy(out, y[perm])
            elif mode == "kd":
                T, a = MIX["kd_T"], MIX["kd_alpha"]; out = model(x)
                kl = F.kl_div(F.log_softmax(out / T, 1), teacher_probs(x, T), reduction="batchmean")
                loss = a * T * T * kl + (1 - a) * F.cross_entropy(out, y)
            else:
                out = model(x); loss = F.cross_entropy(out, y)
            opt.zero_grad(); loss.backward(); opt.step()
        sch.step()
        _, _, tr_loss = evaluate(model, tr_eval)
        yv_, pv_, val_loss = evaluate(model, val_dl); f1v = f1_score(yv_, pv_, average="macro")
        hist.append({"epoch": ep, "train_eval_loss": tr_loss, "val_loss": val_loss, "val_macroF1": f1v, "seconds": round(time.time() - t0, 1)})
        if f1v > best:
            best, best_ep = f1v, ep; torch.save(model.state_dict(), ckpt)
        log(f"epoch {ep:2d} | train-subset loss {tr_loss:.3f} | val loss {val_loss:.3f} macro-F1 {f1v:.3f}{' *' if ep == best_ep else ''} | {time.time() - t0:.0f}s")
    train_time = time.time() - t0
    h = pd.DataFrame(hist); h.to_csv(run_dir / "history.csv", index=False)
    model.load_state_dict(torch.load(ckpt, map_location=device))
    pr, pr_tta = predict_probs(model, PARTS["test"]), predict_probs(model, PARTS["test"], tta=True)
    np.save(Path("test_probs") / f"nb10_{run_id}.npy", pr); np.save(Path("test_probs") / f"nb10_{run_id}_tta.npy", pr_tta)
    m, mt = metrics_from_probs(pr, yt), metrics_from_probs(pr_tta, yt)
    res = dict(run_id=run_id, model=model_name, mode=mode, seed=seed, best_epoch=best_ep, val_f1=best, **m, tta_acc=mt["acc"], tta_f1=mt["f1"],
               val_loss_min=float(h.val_loss.min()), val_loss_last=float(h.val_loss.iloc[-1]),
               val_loss_rise=float(h.val_loss.iloc[-1] - h.val_loss.min()), gap_last=float(h.val_loss.iloc[-1] - h.train_eval_loss.iloc[-1]),
               train_time=train_time, params=n_par)
    log(f"TEST acc {m['acc']:.3f} | macro-F1 {m['f1']:.3f} | 5 groups {m['f1_5']:.3f} | NLL {m['nll']:.3f} | ECE {m['ece']:.3f} | "
        f"power-loss error {m['ploss_err_pct']:+.1f}% | rise {res['val_loss_rise']:.3f} gap {res['gap_last']:.3f} | {train_time:.0f}s")
    config = {"run_id": run_id, "started": started.isoformat(timespec="seconds"), "notebook": NOTEBOOK, "run_folder": str(run_dir), "checkpoint": str(ckpt),
              "model": {"name": model_name, "description": MODEL_INFO[model_name], "trainable_parameters": n_par},
              "fix": mode, "fix_parameters": MIX, "teacher": "mean softmax of R62-R64 (ResNet18S base)" if mode == "kd" else None,
              "training": {"optimizer": "AdamW", "lr": lr, "weight_decay": wd, "epochs": epochs, "lr_schedule": f"CosineAnnealingLR(T_max={epochs})",
                           "batch": batch, "seed": seed, "balancing": "WeightedRandomSampler p ∝ 1/sqrt(n_class)",
                           "augmentation": "H/V flip p=0.5, shift ±2 px", "model_selection": "best validation macro-F1"},
              "environment": environment(),
              "results": {k: (round(float(v), 4) if isinstance(v, (float, np.floating)) else v) for k, v in res.items()}}
    (run_dir / "config.json").write_text(json.dumps(config, indent=2, ensure_ascii=False, default=str), encoding="utf-8")
    log_run(dict(run_id=run_id, model=model_name, recipe=mode, val_acc=float("nan"), val_f1=best, test_acc=m["acc"], test_f1=m["f1"],
                 fault_acc=m["fault_acc"], train_time=train_time, best_epoch=best_ep, val_loss_min=res["val_loss_min"],
                 val_loss_min_epoch=int(h.epoch[h.val_loss.idxmin()]), es_f1=float("nan"), es_nll=float("nan")),
            {"Run folder": str(run_dir), "Model": MODEL_INFO[model_name], "Trainable parameters": f"{n_par:,}", "Fix": mode,
             "Fix parameters": json.dumps(MIX), "Optimizer": f"AdamW, lr = {lr:g}, weight_decay = {wd:g}", "Epochs / batch": f"{epochs} / {batch}",
             "Random seed": str(seed), "Test result": f"acc {m['acc']:.3f}, macro-F1 {m['f1']:.3f}, 5 groups {m['f1_5']:.3f}, NLL {m['nll']:.3f}, ECE {m['ece']:.3f}",
             "Environment": f"PyTorch {torch.__version__}, CUDA {torch.version.cuda}, {device} ({HARDWARE})"},
            note=f"NB10 overfitting fix: {mode}")
    log(f"saved: {run_dir}"); logf.close()
    return res

## 6. Run R131–R139

In [ ]:
PLAN10 = [("R131", "ResNet18S", "mixup", 42), ("R132", "ResNet18S", "mixup", 1), ("R133", "ResNet18S", "mixup", 2),
          ("R134", "ResNet18S", "cutmix", 42), ("R135", "ResNet18S", "cutmix", 1), ("R136", "ResNet18S", "cutmix", 2),
          ("R137", "ResNet10S", "kd", 42), ("R138", "ResNet10S", "kd", 1), ("R139", "ResNet10S", "kd", 2)]
done = set(pd.read_csv("results_10.csv").run_id) if Path("results_10.csv").exists() else set()
t_all = time.time()
for rid, m, mode, sd in PLAN10:
    if rid in done:
        continue
    r = run_fix(rid, m, mode, sd)
    old = pd.read_csv("results_10.csv") if Path("results_10.csv").exists() else pd.DataFrame(columns=COLS10)
    pd.concat([old, pd.DataFrame([r]).reindex(columns=COLS10)], ignore_index=True).to_csv("results_10.csv", index=False)
print(f"all done in {(time.time() - t_all) / 60:.1f} min")

## 7. Comparison with ResNet18S base (R62–R64) and ResNet10S (R65–R67)
Single = mean of 3 seeds; Ensemble = mean of the 3 seeds' TTA probabilities. McNemar: seed-42 model vs R62 on the same 3,000 test images.

In [ ]:
from scipy.stats import binomtest
r8 = pd.read_csv("results_08.csv"); r10 = pd.read_csv("results_10.csv")
SETUPS = [("ResNet18S base (R62–R64)", ["R62", "R63", "R64"], "nb08_{}_f1sel", r8, "f1sel_"),
          ("ResNet10S base (R65–R67)", ["R65", "R66", "R67"], "nb08_{}_f1sel", r8, "f1sel_"),
          ("ResNet18S + mixup (R131–R133)", ["R131", "R132", "R133"], "nb10_{}", r10, ""),
          ("ResNet18S + CutMix (R134–R136)", ["R134", "R135", "R136"], "nb10_{}", r10, ""),
          ("ResNet10S + KD from ensemble (R137–R139)", ["R137", "R138", "R139"], "nb10_{}", r10, "")]
ref = np.load(Path("test_probs") / "nb08_R62_f1sel.npy").argmax(1) == yt
rows = []
for name, ids, pat, tab, pre in SETUPS:
    g = tab[tab.run_id.isin(ids)]
    if len(g) < len(ids):
        print("missing runs for", name); continue
    row = {"setup": name}
    for k in ("acc", "f1", "f1_5", "nll", "ece", "ploss_err_pct"):
        row[k] = g[pre + k].mean()
    row["f1_sd"] = g[pre + "f1"].std(); row["val_loss_rise"] = g.val_loss_rise.mean(); row["gap"] = g.gap_last.mean()
    P = np.mean([np.load(Path("test_probs") / (pat.format(r) + "_tta.npy")) for r in ids], axis=0)
    for k, v in metrics_from_probs(P, yt).items():
        row["ens_" + k] = v
    c = np.load(Path("test_probs") / (pat.format(ids[0]) + ".npy")).argmax(1) == yt
    b, cc = int(np.sum(ref & ~c)), int(np.sum(~ref & c))
    row["mcnemar_p_vs_R62"] = 1.0 if ids[0] == "R62" else binomtest(min(b, cc), b + cc, 0.5).pvalue
    rows.append(row)
comp = pd.DataFrame(rows).round(4); comp.to_csv("results_10_summary.csv", index=False)
pd.set_option("display.width", 260); print(comp.to_string(index=False))

## 8. Fix 5: Label-issue check with confident learning (no training)
Out-of-fold probabilities of ResNet18S from Notebook 09 (every train/val image was held out once in each of the 6 repeats) are averaged per image.
Per-class threshold t_j = mean probability of class j over images labelled j. An image is flagged when its given label's probability is below
t_label and another class j has probability ≥ t_j (the most likely such class is the suggested label). **Flagged ≠ wrong**: these images should be
looked at, not removed automatically.

In [ ]:
POOL = np.concatenate([PARTS["train"], PARTS["val"]])
REPEATS, K = [42, 1, 2, 3, 4, 5], 5
S, N = np.zeros((len(Y), len(CLASSES))), np.zeros(len(Y))
n = 71
for rs in REPEATS:
    for f in range(K):
        idx = np.load(Path("cv_probs") / f"rep{rs}_fold{f}_idx.npy")
        S[idx] += np.load(Path("cv_probs") / f"R{n}.npy"); N[idx] += 1      # R{n} = ResNet18S, R{n+1} = ResNet10S
        n += 2
P_oof = S[POOL] / N[POOL, None]; y_pool = Y[POOL]
t = np.array([P_oof[y_pool == j, j].mean() for j in range(len(CLASSES))])
above = P_oof >= t[None, :]
own_low = P_oof[np.arange(len(y_pool)), y_pool] < t[y_pool]
cand = np.where(above, P_oof, -1); cand[np.arange(len(y_pool)), y_pool] = -1
sugg = cand.argmax(1); flag = own_low & (cand.max(1) > 0)
names = np.array(CLASSES)
issues = pd.DataFrame({"path": split.path.values[POOL][flag], "given": names[y_pool[flag]], "suggested": names[sugg[flag]],
                       "p_given": P_oof[flag, y_pool[flag]].round(3), "p_suggested": P_oof[flag, sugg[flag]].round(3)}).sort_values("p_given")
issues.to_csv("results_10_label_issues.csv", index=False)
per = pd.DataFrame({"class": CLASSES, "images": np.bincount(y_pool, minlength=12), "flagged": np.bincount(y_pool[flag], minlength=12)})
per["flagged_%"] = (100 * per.flagged / per.images).round(1)
print(f"flagged {flag.sum()} of {len(y_pool)} images ({100 * flag.mean():.1f}%)\n"); print(per.to_string(index=False))
print("\nmost common (given -> suggested):"); print(issues.groupby(["given", "suggested"]).size().sort_values(ascending=False).head(10).to_string())

top = issues.head(24)
fig, ax = plt.subplots(3, 8, figsize=(16, 7.5))
for a, (_, r) in zip(ax.ravel(), top.iterrows()):
    a.imshow(np.asarray(Image.open(DATA_DIR / r.path)), cmap="inferno"); a.axis("off")
    a.set_title(f"label: {r.given}\nmodel: {r.suggested} ({r.p_suggested:.2f})", fontsize=7.5)
plt.suptitle("Images whose label the cross-validated model disagrees with most (confident learning)", fontsize=11)
plt.tight_layout(); plt.savefig("figures/fig10_label_issues.png", dpi=200); plt.show()

## 9. Figure for the report: `figures/fig10_fixes_loss.png` (seed 42 loss curves)

In [ ]:
def hist_of(run_id):
    hp = sorted(Path("runs").glob(f"{run_id}_*/history.csv")); return pd.read_csv(hp[-1]) if hp else None
show = [("R62", "ResNet18S base"), ("R131", "ResNet18S + mixup"), ("R134", "ResNet18S + CutMix"), ("R65", "ResNet10S base"), ("R137", "ResNet10S + KD")]
fig, ax = plt.subplots(1, len(show), figsize=(20, 4.2), sharey=True)
for a, (rid, title) in zip(ax, show):
    h = hist_of(rid)
    if h is None: a.set_title(f"{rid}: not found"); continue
    a.plot(h.epoch, h.train_eval_loss, color="#2E5E8C", lw=2, label="train subset"); a.plot(h.epoch, h.val_loss, color="#8A1F2B", lw=2, label="validation")
    a.fill_between(h.epoch, h.train_eval_loss, h.val_loss, color="#8A1F2B", alpha=0.08)
    a.set_title(f"{title} ({rid})\nend gap {h.val_loss.iloc[-1] - h.train_eval_loss.iloc[-1]:.2f}", fontsize=10); a.set_xlabel("epoch"); a.grid(alpha=0.25)
ax[0].set_ylabel("cross-entropy loss"); ax[0].set_ylim(0); ax[0].legend(frameon=False)
plt.tight_layout(); plt.savefig("figures/fig10_fixes_loss.png", dpi=200); plt.show()

## References
* C. Guo, G. Pleiss, Y. Sun, K. Q. Weinberger, "On Calibration of Modern Neural Networks," ICML 2017.
* H. Zhang, M. Cisse, Y. N. Dauphin, D. Lopez-Paz, "mixup: Beyond Empirical Risk Minimization," ICLR 2018.
* S. Thulasidasan et al., "On Mixup Training: Improved Calibration and Predictive Uncertainty for Deep Neural Networks," NeurIPS 2019.
* S. Yun et al., "CutMix: Regularization Strategy to Train Strong Classifiers with Localizable Features," ICCV 2019.
* G. Hinton, O. Vinyals, J. Dean, "Distilling the Knowledge in a Neural Network," arXiv:1503.02531, 2015.
* C. Northcutt, L. Jiang, I. Chuang, "Confident Learning: Estimating Uncertainty in Dataset Labels," JAIR 70, 2021.